In [1]:
import numpy as np
import csv
from pathlib import Path


# import matplotlib.pyplot as plt

# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression, RidgeClassifierCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
# from sklearn.pipeline import make_pipeline

# import pandas as pd

# import pycatch22
# from sktime.transformations.rocket import MiniRocket

In [2]:
def norm(signal):
    return (signal-np.mean(signal))/np.std(signal)

In [3]:
# Noise Processes
def gauss_noise(N,mu=0,sigma=1):
    """
    Gaussian distributed noise. Default x~N(0,1).
    """
    return np.random.normal(mu,sigma,N)

def uniform_noise(N,a=-0.5,b=0.5):
    """
    Uniformly distributed noise. Default x~U(-0.5,0.5).
    """
    return np.random.uniform(a,b,N)

def pink_noise(N,mu=0,sigma=1):
    """
    Pink noise with power density S(f) p.t. 1/f.
    """
    x = gauss_noise(N,mu,sigma)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(N)
    f[0] = f[1]
    X = X/np.sqrt(f)

    pink = np.fft.irfft(X,n=N)
    pink = norm(pink)
    return pink

def red_noise(N,a=0.9,mu=0,sigma=1):
    """
    Generated via an AR(1) process. Default a=0.9.
    """
    noise = gauss_noise(N,mu,sigma)
    x = np.zeros(N)
    for t in range(1,N):
        x[t] = a*x[t-1]+noise[t]
    x = norm(x)
    return x

def violet_noise(N,mu=0,sigma=1):
    """
    Violet noise with power density S(f) p.t. f^2.
    Generated by differentiating white noise.
    """
    white = gauss_noise(N,mu,sigma)
    violet = np.diff(white,prepend=white[0])
    violet = norm(violet)
    return violet

def gamma_noise(N,alpha=1,theta=0.3):
    """
    Gamma distribution noise. Default x~Gamma(1,0.3).
    """
    return np.random.gamma(alpha,theta,N)

def laplace_noise(N,mu=0,sigma=1):
    """
    Laplacian distribution noise. Default x~Laplace(0,1).
    """
    return np.random.laplace(mu,sigma,N)

In [4]:
# Autoregressive models
def AR1(N,a=0.5,mu=0,sigma=1):
    """
    First order autoregressive model AR(1). Default a=0.5.
    """
    x = np.zeros(N)
    noise = gauss_noise(N,mu,sigma)
    
    for t in range(1,N):
        x[t] = a*x[t-1]+noise[t]
    return x

def static_AR1(N, a=0.6, mu=0,sigma=1):
    """
    AR(1) model with static transformation y=tanh^2(x). Default a=0.6.
    """
    x = AR1(N,a,mu,sigma)
    y = np.tanh(x)**2
    return y

def uniform_AR1(N,alpha=0.5,a=-0.5,b=0.5,sigma=1):
    """
    Linear AR(1) model with uniformly sampled noise rather than Gaussian. 
    Default again alpha=0.5.
    """
    x = np.zeros(N)
    noise = uniform_noise(N,a*sigma,b*sigma)

    for t in range(1,N):
        x[t] = alpha*x[t-1]+noise[t]
    return x

def ARMA11(N,alpha=0.6,theta=0.4,a=-0.5,b=0.5,sigma=1):
    """
    Linear ARMA(1,1) model with uniformly sampled noise. 
    Default alpha=0.6, theta=0.4.
    """
    x = np.zeros(N)
    noise = uniform_noise(N,a*sigma,b*sigma)

    for t in range(1,N):
        x[t] = alpha*x[t-1] + noise[t] + theta*noise[t-1]
    return x

def AR3_gamma(N,a=0.1,b=-0.2,c=0.3,alpha=1,theta=0.3):
    """
    Third order autoregressive model AR(3) with gamma noise.
    Default a=0.1, b=0.2, c=0.3, x~Gamma(1,0.3).
    """
    x = np.zeros(N)
    noise = gamma_noise(N,alpha,theta)

    for t in range(3,N):
        x[t] = c*x[t-3]+b*x[t-2]+a*x[t-1]+noise[t]
    return x

def SETAR2111(N,a1=-0.9,a2=-0.4,theta1=1,theta2=1,threshold=1,mu=0,sigma=1):
    """
    Self-exciting threshold autoregressive model SETAR(2;1,1,1),
    with two AR(1) regimes, switching controlled by the first delay paramater and both systems first order.
    Default a1=-0.9, a2=-0.4, theta=1, threshold x[t-1]=1.
    """
    x = np.zeros(N)
    noise = gauss_noise(N,mu,sigma)

    for t in range(1,N):
        if x[t-1]>=threshold:
            x[t]=a1*x[t-1]+theta1*noise[t]
        else:
            x[t]=a2*x[t-1]+theta2*noise[t]
    return x

def SETAR2222(N,c1=0.62,a1=1.25,b1=-0.43,theta1=0.0381,
              c2=2.25,a2=1.52,b2=-1.24,theta2=0.0626,
              threshold=3.25,mu=0,sigma=1):
    """
    SETAR(2;2,2,2) model, with two AR(2) regimes, switching controlled by the second delay paramater.
    Defaults as given.
    """
    x = np.zeros(N)
    noise = gauss_noise(N,mu,sigma)

    for t in range(1,N):
        if x[t-2]<=threshold:
            x[t]=c1+a1*x[t-1]+b1*x[t-2]+theta1*noise[t]
        else:
            x[t]=c2+a2*x[t-1]+b2*x[t-2]+theta2+noise[t]
    return x

def nl_AR2(N,w1=0.5,a1=-0.3,alpha1=0,beta1=0.1,
           w2=0,a2=0.1,alpha2=0.4,beta2=0,eta=0.0025,
           k1=4,k2=6,xi=0.0025,mu0=0,sigma0=1,
           n1=-0.5,mu1=0.63,sigma1=1,n2=0.5,mu2=-0.63,sigma2=1):
    """
    Nonlinear AR(2) process. Consists of various quadratic and sinusoidal terms as seen below.
    Defaults as given.
    """
    x = np.zeros(N)
    y = np.zeros(N)
    eta_noise = laplace_noise(N,mu0,sigma0)
    xi_noise = n1*gauss_noise(N,mu1,sigma1) + n2*gauss_noise(N,mu2,sigma2)

    for t in range(1,N-1):
        y[t] = np.sin(k1*np.pi*t) + np.sin(k2*np.pi*t) + xi*xi_noise[t]
        x[t] = (w1*x[t] + a1*x[t-1] + alpha1*x[t]**2 + beta1*x[t-1]**2 + 
                w2*y[t] + a2*y[t-1] + alpha2*x[t]**2 + beta2*x[t-1]**2 + 
                eta*eta_noise[t])
    return x

In [5]:
# Deterministic chaotic maps
def rand_init(l=0,r=1):
    """
    HELPER
    Randomised uniform initial conditions for deterministic maps. Samples randomly from the interval [0,1).
    """
    return np.random.uniform(l,r)

def arnold_cat(N,l=0,r=1,a1=1,b1=1,a2=1,b2=2,n=1,use_x=True):
    """
    Arnold's Cat Map of the form x[t+1] = a1x[t] + b1y[t] mod n, y[t+1] = a2x[t] + b2y[t] mod n.
    Defaults as given. Consider by default the x component if use_x=True, y component is use_x=False.
    """
    x = np.zeros(N)
    y = np.zeros(N)
    x[0] = rand_init(l,r)
    y[0] = rand_init(l,r)
    for t in range(1,N):
        x[t] = (a1*x[t-1] + b1*y[t-1]) % n
        y[t] = (a2*x[t-1] + b2*y[t-1]) % n
    if use_x==False:
        return y
    return x

def chirikov(N,l=0,r=1,K=0.971635,n=1,use_p=True):
    """
    Chikrov standard map of the form theta[t+1] = theta[t] + p[t] + K/2pi sin(2pi theta[t]) mod n,
    p[t+1] = theta[t+1] - theta[t] mod n. Consider p component by default, theta if use_p=False.
    """
    theta = np.zeros(N)
    p = np.zeros(N)
    theta[0] = rand_init(l,r)
    p[0] = rand_init(l,r)
    for t in range(1,N):
        theta[t] = (theta[t-1] + p[t-1] + K/(2*np.pi)*np.sin(2*np.pi*theta[t-1])) % n
        p[t] = (theta[t] - theta[t-1]) % n
    if use_p==False:
        return theta
    return p

def henon(N,l=0,r=1,a=1.4,b=0.3,use_x=True):
    """
    Standard Henon Map x[t+1]=1-ax[t]^2+y[t], y[t+1] = bx[t].
    Consider x by default.
    """
    x = np.zeros(N)
    y = np.zeros(N)
    x[0] = rand_init(l,r)
    y[0] = rand_init(l,r)
    for t in range(1,N):
        x[t] = 1 - a*x[t-1]**2 + y[t-1]
        y[t] = b*x[t-1]
    if use_x==False:
        return y
    return x

def quad(N,l=0,r=1,a=1.8):
    """
    Quadratic map with scaling factor a of the form x[t+1]=1-ax[t]^2.
    """
    x = np.zeros(N)
    x[0] = rand_init(l,r)
    for t in range(1,N):
        x[t] = 1-a*x[t-1]**2
    return x

def logistic(N,R,l=0,r=1):
    """
    HELPER
    Logistic map with logistic factor R of the form x[t+1]=rx[t](1-x[t]).
    """
    x = np.zeros(N)
    x[0] = rand_init(l,r)
    for t in range(1,N):
        x[t] = R*x[t-1]*(1-x[t-1])
    return x

def logistic_4(N,l=0,r=1):
    """
    Logistic map with R=4, which lies in the fully chaotic regime.
    """
    return logistic(N,4,l,r)

def logistic_38(N,l=1,r=1):
    """
    Logistic map with R=3.8284, which lies in the period-3 window, where it exhibits intermittency.
    """
    return logistic(N,3.8284,l,r)


In [6]:
# Sums of deterministic chaotic maps/flows
def sum_henon_fr(N,l=0,r=1,a=1.4,b=0.3):
    """
    Sum of x components of two independent realisations of the Henon map, with one taken in forward time and the other in reverse.
    """
    forward = henon(N,l,r,a,b)
    reverse = henon(N,l,r,a,b)[::-1]

    return np.mean([forward,reverse])

def sum_henon_fr_eq(N,l=0,a=1.4,b=0.3):
    """
    Sum of x components of the forward and reverse time realistations of a single Henon map realisation.
    """
    forward = henon(N,l,r,a,b)
    reverse = forward[::-1]

    return np.mean([forward,reverse])

def sum_quad_fr(N,l=0,r=1,a=1.8):
    """
    Sum of x components of two independent realisations of the Quadratic map, with one in forward time and the eother in reverse.
    """
    forward = quad(N,l,r,a)
    reverse = quad(N,l,r,a)[::-1]

    return np.mean([forward,reverse])

def sum_henon(N,l=0,r=1,a=1.4,b=0.3):
    """
    Sum of two independenr forward realisations of x components of the Henon map.
    """
    map1 = henon(N,l,r,a,b)
    map2 = henon(N,l,r,a,b)

    return np.mean([map1,map2])

def sum_lorenz(N,lx=-8,rx=8,ly=-8,ry=8,lz=0,rz=10,sigma=10,rho=28,beta=8/3,dt=0.01):
    """
    Sum of three independent Lorenz systems with equations dx/dt=sigma(y-x), dy/dt=x(rho-z)-y, dz/dt=xy-betaz.
    """
    x = np.zeros(N)
    y = np.zeros(N)
    z = np.zeros(N)

    x[0] = np.random.uniform(lx, rx)
    y[0] = np.random.uniform(ly,ry)
    z[0] = np.random.uniform(lz,rz)

    def derivatives(state):
        x, y, z = state
        dx = sigma * (y - x)
        dy = x * (rho - z) - y
        dz = x * y - beta * z
        return np.array([dx, dy, dz])

    # RK4 integration
    for i in range(N - 1):
        state = np.array([x[i], y[i], z[i]])

        k1 = derivatives(state)
        k2 = derivatives(state + dt * k1 / 2)
        k3 = derivatives(state + dt * k2 / 2)
        k4 = derivatives(state + dt * k3)

        next_state = state + dt / 6 * (k1 + 2*k2 + 2*k3 + k4)

        x[i+1], y[i+1], z[i+1] = next_state

    return np.mean([x, y, z])

def sum_rossler(N,lx=-8,rx=8,ly=-8,ry=8,z0=0,a=0.2,b=0.2,c=5.7,dt=0.01):
    """
    Sum of three independent Rossler systems with dx/dt=-y-z, dy/dt=x+ay, dz/dt=b+z(x-c).
    """
    x = np.zeros(N)
    y = np.zeros(N)
    z = np.zeros(N)

    x[0] = np.random.uniform(lx, rx)
    y[0] = np.random.uniform(ly, ry)
    z[0] = z0

    def derivatives(state):
        x, y, z = state
        dx = -y - z
        dy = x + a*y
        dz = b + z*(x - c)
        return np.array([dx, dy, dz])

    # RK4 integration
    for i in range(N - 1):
        state = np.array([x[i], y[i], z[i]])

        k1 = derivatives(state)
        k2 = derivatives(state + dt*k1/2)
        k3 = derivatives(state + dt*k2/2)
        k4 = derivatives(state + dt*k3)

        next_state = state + dt*(k1 + 2*k2 + 2*k3 + k4)/6

        x[i+1], y[i+1], z[i+1] = next_state

    return np.mean([x, y, z])

In [7]:
# Other deterministic models
def linear_log(N,a=0.5,b=4):
    """
    Linear model x[t+1] = ax[t] + epsilon[t] 
    with deterministic logistic noise source epsilon[t] = 4*epsilon[t-1](1-epsilon[t-1]).
    """
    x = np.zeros(N)
    epsilon = np.zeros(N)

    for t in range(1,N-1):
        epsilon[t] = b*epsilon[t-1]*(1-epsilon[t-1])
        x[t+1] = a*x[t]+epsilon[t]

    return x

def mod1(N,a=np.sqrt(2)):
    """
    Modulo-1 model x[t] = ax[t-1] mod 1
    """
    x = np.zeros(N)

    for t in range(1,N):
        x[t] = a*x[t-1] % 1

    return x

def lin_oscillator(N,lx=-2,rx=2,ly=-2,ry=2,dt=0.01):
    """
    Linearised oscillator with dynamics described by d^2x/dt^2 + x = 0, y = dx/dt, dy/dt=-x.
    """
    x = np.zeros(N)
    y = np.zeros(N)

    x[0] = np.random.uniform(lx,rx)
    y[0] = np.random.uniform(ly,ry)

    def derivatives(state):
        x, y = state
        return np.array([y, -x])

    for i in range(N - 1):
        state = np.array([x[i], y[i]])

        k1 = derivatives(state)
        k2 = derivatives(state + dt*k1/2)
        k3 = derivatives(state + dt*k2/2)
        k4 = derivatives(state + dt*k3)

        state_next = state + dt*(k1 + 2*k2 + 2*k3 + k4)/6
        x[i+1], y[i+1] = state_next

    return x

def dvp_oscillator(N,lx=-2,rx=2,ly=-2,ry=2,mu=1,dt=0.01):
    """
    Van der Pol oscillator, specified by d^2x/dt^2 - mu(1-x^2)dx/dt + x = 0, y=dx/dt, dy/dt = mu(1-x^2)y - x.
    Default value mu=1 includes nonlinear damping.
    """
    x = np.zeros(N)
    y = np.zeros(N)

    x[0] = np.random.uniform(lx,lr)
    y[0] = np.random.uniform(ly,ry)

    def derivatives(state):
        x, y = state
        dx = y
        dy = mu*(1 - x**2)*y - x
        return np.array([dx, dy])

    for i in range(N - 1):
        state = np.array([x[i], y[i]])

        k1 = derivatives(state)
        k2 = derivatives(state + dt*k1/2)
        k3 = derivatives(state + dt*k2/2)
        k4 = derivatives(state + dt*k3)

        state_next = state + dt*(k1 + 2*k2 + 2*k3 + k4)/6
        x[i+1], y[i+1] = state_next

    return x

def mackey_glass(N,l=0,r=1.5,beta0=0.2,gamma=0.1,n=10,tau=17,dt=1):
    """
    Mackey-Glass temporal delay equation, given by dx/dt = (beta0*x(t-tau))/(1+x(t-tau)^n) - gamma*x(t).
    """
    x = np.zeros(N)
    x[0] = np.random.uniform(0, 1.5)

    delay_steps = int(tau / dt)

    # Initialise the history before t = 0
    x[:min(delay_steps, N)] = x[0]

    for i in range(delay_steps, N):
        x_tau = x[i - delay_steps]

        dx = beta0 * x_tau / (1 + x_tau**n) - gamma*x[i]

        x[i] = x[i-1] + dt*dx

    return x

In [8]:
# Other stochastic processes
def stochastic_sine(N,mu=2.4,p=0.01,a=-2,b=2):
    """
    Stochastic noise-driven sine map defined by x[t+1] = mu*sin(x[t]) + y[t]*eta[t],
    with y[t]~Ber(p), eta[t]~U(a,b).
    """
    x = np.zeros(N)
    y = np.random.binomial(1,p,N)
    eta = np.random.uniform(a,b,N)

    for t in range(N-1):
        x[t+1] = mu*sin(x[t]) + y[t]*eta[t]

    return x

def weiner(dt):
    """
    HELPER
    Weiner process simulated via Euler-Maruyama method, dW_t = sqrt(dt)*epsilon_t, with epsilon_t~N(0,1).
    """
    return np.sqrt(dt) * np.random.normal()

def ornstein_uhlenbeck(N,theta=0.8,sigma=0.3,x0=0,dt=0.01):
    """
    Ornstein-Uhlenbeck process given by dx = -theta*xdt + sigma*dW, where W is a Weiner process.
    """
    x = np.zeros(N)
    x[0] = x0

    for t in range(N - 1):
        dW = weiner(dt)
        dx = -theta*x[t]*dt + sigma*dW
        x[t+1] = x[t] + dx

    return x

def BRW(N,k=-2,alpha1=2,alpha2=2,sigma=4,beta=0.1,x0=0,tau=0,dt=0.001):
    """
    Bounded Random Walk defined by dx = e^k(e^(-alpha1(x-tau))=e^(alpha2(x-tau)))*dt + e^(sigma/2 + 2x[t]^2)*dW.
    """
    x = np.zeros(N)
    x[0] = x0

    for t in range(N-1):
        dW = weiner(dt)
        drift = np.exp(k)*(np.exp(-alpha1*(x[t]-tau)) + np.exp(alpha2*(x[t]-tau)))
        diffusion = np.exp(sigma/2 + beta/2*x[t]**2)
        dx = drift*dt + diffusion*dW
        x[t+1] = x[t] + dx

    return x

def stochastic_lorenz(N,lx=-8,rx=8,ly=-8,ry=8,lz=0,rz=10,sigma=10,rho=28,beta=8/3,noise=1.5,dt=0.01):
    """
    Sum of three independent Lorenz systems with equations dx/dt=sigma(y-x), dy/dt=x(rho-z)-y, dz/dt=xy-betaz 
    with additive Weiner processes attached to each element of the process.
    """
    x = np.zeros(N)
    y = np.zeros(N)
    z = np.zeros(N)

    x[0] = np.random.uniform(lx,rx)
    y[0] = np.random.uniform(ly,ry)
    z[0] = np.random.uniform(lz,rz)

    for t in range(N-1):
        dx = sigma*(y[t]-x[t])
        dy = x[t]*(rho-z[t]) - y[t]
        dz = x[t]*y[t] - beta*z[t]

        dWx = weiner(dt)
        dWy = weiner(dt)
        dWz = weiner(dt)

        x[t+1] = x[t] + dx*dt + noise*dWx
        y[t+1] = y[t] + dy*dt + noise*dWy
        z[t+1] = z[t] + dz*dt + noise*dWz

    return np.mean([x, y, z])

def stochastic_rossler(N,lx=-8,rx=8,ly=-8,ry=8,z0=0,a=0.2,b=0.2,c=5.7,noise=1.5,dt=0.01):
    """
    Sum of three independent Rossler systems with dx/dt=-y-z, dy/dt=x+ay, dz/dt=b+z(x-c).
    """
    x = np.zeros(N)
    y = np.zeros(N)
    z = np.zeros(N)

    x[0] = np.random.uniform(lx, rx)
    y[0] = np.random.uniform(ly, ry)
    z[0] = z0

    for t in range(N-1):
        dx = -y[t] - z[t]
        dy = x[t] + a*y[t] 
        dz = b + z[t]*(x[t]-c)

        dWx = weiner(dt)
        dWy = weiner(dt)
        dWz = weiner(dt)
        
        x[t+1] = x[t] + dx*dt + noise*dWx
        y[t+1] = y[t] + dy*dt + noise*dWy
        z[t+1] = z[t] + dz*dt + noise*dWz

    return np.mean([x, y, z])

In [9]:
functions_all = [
    gauss_noise, uniform_noise, pink_noise, red_noise, violet_noise, gamma_noise, laplace_noise,
    AR1, static_AR1, uniform_AR1, ARMA11, AR3_gamma, SETAR2111, SETAR2222, nl_AR2,
    arnold_cat, chirikov, henon, quad, logistic_4, logistic_38,
    sum_henon_fr, sum_henon_fr_eq, sum_quad_fr, sum_henon, sum_lorenz, sum_rossler,
    linear_log, mod1, lin_oscillator, dvp_oscillator, mackey_glass,
    stochastic_sine, ornstein_uhlenbeck, BRW, stochastic_lorenz, stochastic_rossler
]
time_reversible_all = [
    True, True, True, True, True, True, True,
    True, True, False, False, False, False, False, False,
    True, True, False, False, False, False,
    True, True, True, False, False, False,
    False, False, True, False, False,
    False, True, True,  False, False
]

discrete_all = [
    True, True, True, True, True, True, True,
    True, True, True, True, True, True, True, True,
    True, True, True, True, True, True,
    True, True, True, True, False, False,
    True, True, False, False, False,
    True, False, False, False, False
]

labels_all = [
    'Gaussian noise', 'Uniform noise', 'Pink noise', 'Red noise', 'Violet noise', 'Gamma noise', 'Laplacian noise',
    'AR(1) with Gaussian noise', 'Static trasformation of AR(1)', 'AR(1) with uniform noise', 'ARMA(1,1) with uniform noise',
    'AR(3) with gamma sampled noise', 'SETAR(2;1,1,1)', 'SETAR(2;2,2,2)', 'Nonlinear AR(2) with Laplacian noise',
    "Arnold's Cat map", 'Chirikov standard map', 'Henon map', 'Quadratic map', 'Chaotic logistic map', 'intermittent logistic map',
    'Sum of forward and reverse Henon maps', 'Sum of Henon map and its reverse', 'Sum of forward and reverse quadratic map',
    'Sum of two Henon maps', 'Sum of three Lorenz flows', 'Sum of three Rossler flows',
    'Linear model with logistic noise', 'Modulo-1 model', 'Linear oscillator', 'Van der Pol oscillator', 'Mackey-Glass model',
    'Stochastic sine map', 'Ornstein-Uhlenbeck process', 'Bounded Random Walk (BRW)', 'Stochastic Lorenz flows', 
    'Stochastic Rossler flows'
]

print(len(functions_all), len(time_reversible_all), len(discrete_all), len(labels_all))


37 37 37 37


In [10]:
# Edited lists to remove problems because I simply can't be arsed
functions = [
    gauss_noise, uniform_noise, pink_noise, red_noise, violet_noise, gamma_noise, laplace_noise,
    AR1, static_AR1, uniform_AR1, ARMA11, AR3_gamma, SETAR2111, SETAR2222, nl_AR2,
    arnold_cat, chirikov, quad, logistic_4, logistic_38,
    
    linear_log, mod1, lin_oscillator, mackey_glass,
     ornstein_uhlenbeck, 
]
time_reversible = [
    True, True, True, True, True, True, True,
    True, True, False, False, False, False, False, False,
    True, True, False, False, False,
    
    False, False, True, False,
    True, 
]

discrete = [
    True, True, True, True, True, True, True,
    True, True, True, True, True, True, True, True,
    True, True, True, True, True,
    
    True, True, False, False,
    False 
]

labels = [
    'Gaussian noise', 'Uniform noise', 'Pink noise', 'Red noise', 'Violet noise', 'Gamma noise', 'Laplacian noise',
    'AR(1) with Gaussian noise', 'Static trasformation of AR(1)', 'AR(1) with uniform noise', 'ARMA(1,1) with uniform noise',
    'AR(3) with gamma sampled noise', 'SETAR(2;1,1,1)', 'SETAR(2;2,2,2)', 'Nonlinear AR(2) with Laplacian noise',
    "Arnold's Cat map", 'Chirikov standard map', 'Quadratic map', 'Chaotic logistic map', 'intermittent logistic map',
    
    'Linear model with logistic noise', 'Modulo-1 model', 'Linear oscillator', 'Mackey-Glass model',
    'Ornstein-Uhlenbeck process'
]

print(len(functions), len(time_reversible), len(discrete), len(labels))

    

25 25 25 25


In [11]:
tests = []
tests_reversible = []
tests_label = []

N = 500
n_tests = 1000

failed = []
nan = []

for i in range(len(functions)):
    function = functions[i]
    reversible = time_reversible[i]
    disc = discrete[i]
    label = labels[i]

    for _ in range(n_tests):

        try:
            if disc:
                test = function(N)
            else:
                test = function(2*N)
                test = test[N:]

            test = np.asarray(test, dtype=float)

            if test.shape != (N,) or not np.all(np.isfinite(test)):
                if not np.isin(function,failed):
                    failed.append(function)
                continue
            # if np.isin(test,NaN):
            #     if not np.isin(function,nan):
            #         nan.append(function)

            tests.append(test)
            tests_reversible.append(reversible)
            tests_label.append(label)

        except Exception:
            if not np.isin(function,failed):
                failed.append(function)
            continue

print("Failed simulations:")
print(failed)

# Shuffle order and check each block of 50 has at least one reversible and one irreversible
block_length = 50

assert len(tests) % block_length == 0

tests = np.asarray(tests)
tests_reversible = np.asarray(tests_reversible).astype(int)
tests_label = np.asarray(tests_label)

while True:
    order = np.random.permutation(len(tests))

    shuffled_reversible = tests_reversible[order]
    blocks = shuffled_reversible.reshape(-1, block_length)

    # Every block must contain both True and False
    valid = all(
        np.any(block) and np.any(~block)
        for block in blocks
    )

    if valid:
        tests = tests[order]
        tests_reversible = tests_reversible[order]
        tests_label = tests_label[order]
        break
            
    
            
        

Failed simulations:
[]


In [12]:
def gen_csv(tests, labels, conditions, path="data.csv"):
    assert len(tests) == len(labels) == len(conditions)
    assert tests.ndim == 2

    length = tests.shape[1]

    with Path(path).open("w", newline="") as handle:
        writer = csv.writer(handle)

        writer.writerow(['Condition', 'Label'] + [f't{i}' for i in range(length)])

        for condition,label,test in zip(conditions,labels,tests):
            writer.writerow([condition, label] + [f'{value:.6f}' for value in test])
            # Checker with blank condition 'Condition'
            # writer.writerow(['Condition',label] + [f'{value:.6f}' for value in test])

    return Path(path)

gen_csv(tests, tests_reversible, tests_label)
            

PosixPath('data.csv')